# SI7016 — Clase 03 — Lab B
## RAG con Hugging Face + Qdrant

**Objetivo:** separar y evaluar retrieval y generation.

In [ ]:
# %pip install -q "qdrant-client>=1.15,<2" "sentence-transformers>=5,<6" langchain-anthropic langchain-openai python-dotenv
import os
from dotenv import load_dotenv
load_dotenv()
documentos=[
"MCP estandariza la conexión con tools, resources y prompts.",
"A2A permite interacción y delegación entre agentes.",
"LangGraph modela flujos con estado y checkpointing.",
"RAG combina recuperación de información con generación."
]

## 1. Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
emb=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vecs=emb.encode(documentos,normalize_embeddings=True)
print(vecs.shape)

## 2. Qdrant

In [ ]:
from qdrant_client import QdrantClient, models
client=QdrantClient(":memory:")
COL="si7016_class03"
client.create_collection(COL,vectors_config=models.VectorParams(size=vecs.shape[1],distance=models.Distance.COSINE))
client.upsert(COL,points=[models.PointStruct(id=i,vector=vecs[i].tolist(),payload={"texto":d}) for i,d in enumerate(documentos)])
print(client.count(COL).count)

## 3. Retrieval con `query_points()`

In [ ]:
def recuperar(pregunta,k=2):
    q=emb.encode(pregunta,normalize_embeddings=True).tolist()
    res=client.query_points(collection_name=COL,query=q,limit=k,with_payload=True)
    return [(p.score,p.payload["texto"]) for p in res.points]
print(recuperar("¿Qué protocolo permite delegar entre agentes?"))

## 4. Generación fundamentada

In [ ]:
def prompt_rag(pregunta,k=2):
    contexto="\n".join("- "+t for _,t in recuperar(pregunta,k))
    return f"Responde SOLO con base en el contexto. Si no basta, di que no hay evidencia suficiente.\nContexto:\n{contexto}\nPregunta: {pregunta}"
def generar(pregunta):
    p=prompt_rag(pregunta)
    if os.getenv("ANTHROPIC_API_KEY"):
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=os.getenv("ANTHROPIC_MODEL","claude-sonnet-4-6")).invoke(p).content
    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=os.getenv("OPENAI_MODEL","gpt-5.6")).invoke(p).content
    return p
print(generar("¿Para qué sirve A2A?"))

## 5. Evaluar retrieval

In [ ]:
casos=[("¿Qué usa checkpointing?","LangGraph"),("¿Qué combina recuperación y generación?","RAG"),("¿Qué conecta tools?","MCP")]
for q,e in casos:
    top=recuperar(q,1)[0][1]
    print("OK" if e.lower() in top.lower() else "REVISAR","|",q,"=>",top)

## Ejercicio
1. Use un PDF real con `pypdf`. 2. Compare chunk sizes. 3. Agregue re-ranking. 4. Use Qdrant en Docker. 5. Defina 10 preguntas y calcule Recall@k.